# Fase 1 · Paso 4: regresión histórica MCR-2026-004

Candidato determinista fijado de Emory-HITI con adaptación DICOM. Ejecuta exclusivamente las mismas diez adquisiciones (20 rodillas) del piloto histórico, tras comprobar el cierre rechazado de v0.4 en Colab. Produce candidatas y hoja de revisión técnica. **No aprueba recortes ni selecciona participantes nuevos.** No procesa masivamente, crea particiones, entrena ni abre la prueba reservada.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
private_root = userdata.get('KNEE_DATA_ROOT') or os.environ.get('KNEE_DATA_ROOT')
if not private_root:
    raise RuntimeError('Configura KNEE_DATA_ROOT como secreto privado de Colab.')
os.environ['KNEE_DATA_ROOT'] = private_root
private_root_path = Path(private_root).resolve()
step4_dir = private_root_path/'outputs'/'auditorias'/'fase_1'/'paso_4_localizacion_tibiofemoral'
v04_dir = step4_dir/'v0_4_piloto'
for name in ('resumen_revision_visual_publico.json', 'registro_cierre_revision.json'):
    if not (v04_dir/name).is_file():
        raise FileNotFoundError(f'Falta el cierre v0.4: {name}')
print('Drive y cierre v0.4 disponibles.')

## Obtener la revisión publicada del repositorio

Esta libreta no puede ejecutarse hasta que el código MCR-2026-004 haya sido autorizado y publicado en la rama actual. No usará una copia local no versionada.

In [ ]:
REPO_URL = 'https://github.com/AlonsoYess/Knee.git'
BRANCH = 'codex/fase-0-base-reproducible'
REPO_DIR = Path('/content/Knee_mcr004_regresion_historica')
if REPO_DIR.exists():
    raise FileExistsError('El clon ya existe: utiliza una sesión Colab nueva para evitar mezclar revisiones.')
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, check=True,
                        capture_output=True, text=True).stdout.strip()
required = REPO_DIR/'src'/'knee'/'roi_mcr004_pilot.py'
if not required.is_file():
    raise RuntimeError('La rama publicada aún no contiene MCR-2026-004; detenerse.')
print('Revisión de código:', commit)

## Instalar entorno compatible y ejecutar pruebas antes del piloto

Usar Colab 2026.07 con Python 3.12. La versión 1.1 usa `virtualenv` fijado solo para crear el entorno aislado: evita el fallo de `ensurepip` observado en Colab, sin cambiar las dependencias científicas ni el algoritmo.

In [ ]:
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Esta libreta requiere Python 3.12 (Colab 2026.07).')
VENV_DIR = Path('/content/venv_mcr004')
if VENV_DIR.exists():
    raise FileExistsError('El entorno MCR004 ya existe; utiliza una sesión Colab nueva.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv==21.7.11'], check=True)
created = subprocess.run([sys.executable, '-m', 'virtualenv', '--no-periodic-update',
                          str(VENV_DIR)], capture_output=True, text=True)
if created.returncode:
    raise RuntimeError(f'No se pudo crear el entorno aislado ({created.returncode}): '                       f'{created.stderr or created.stdout}')
venv_python = VENV_DIR/'bin'/'python'
subprocess.run([str(venv_python), '-m', 'pip', '--version'], check=True)
subprocess.run([str(venv_python), '-m', 'pip', 'install', '-q',
                '-r', str(REPO_DIR/'requirements-mcr004.txt')], check=True)
subprocess.run([str(venv_python), '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
subprocess.run([str(venv_python), '-m', 'unittest', 'discover', '-s', 'tests',
                '-p', 'test_roi_mcr004*.py', '-v'], cwd=REPO_DIR, check=True)
version_code = "import json,sys,cv2,numpy,scipy,pydicom;print(json.dumps({'python':sys.version.split()[0],'numpy':numpy.__version__,'scipy':scipy.__version__,'opencv':cv2.__version__,'pydicom':pydicom.__version__}))"
environment = json.loads(subprocess.run([str(venv_python), '-c', version_code], check=True,
                                    capture_output=True, text=True).stdout)
environment['virtualenv_bootstrap'] = '21.7.11'
print('Entorno aislado:', environment)

## Ejecutar solo la regresión histórica

La ejecución comprueba ambos JSON de cierre de v0.4, la auditoría DICOM y la separación bilateral congelada. Si existe una salida anterior no la sobrescribe.

In [ ]:
config_path = REPO_DIR/'configs'/'roi_mcr004_historical.example.json'
config = json.loads(config_path.read_text(encoding='utf-8'))
output_dir = (private_root_path/config['output_dir']).resolve()
subprocess.run([str(venv_python), '-m', 'knee.roi_mcr004_pilot',
                '--config', str(config_path)], cwd=REPO_DIR, check=True)
summary_path = output_dir/'resumen_regresion_publico.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
import hashlib
from datetime import datetime, timezone
record = {'phase': 1, 'step': 4, 'operation': 'prepare_mcr004_historical_regression',
          'executed_at_utc': datetime.now(timezone.utc).isoformat(),
          'git_commit': commit, 'environment': environment,
          'summary_sha256': hashlib.sha256(summary_path.read_bytes()).hexdigest(),
          'review_template_sha256': hashlib.sha256((output_dir/'revision_tecnica_ciega.csv').read_bytes()).hexdigest(),
          'prior_v04_closure': summary['prior_v04_closure'],
          'candidate_knees': summary['candidate_knees'], 'abstained_knees': summary['abstained_knees'],
          'visual_review_status': 'pending', 'mass_processing_executed': False,
          'partitions_created': False, 'training_executed': False, 'reserved_test_opened': False}
with (output_dir/'registro_preparacion.json').open('x', encoding='utf-8') as stream:
    json.dump(record, stream, ensure_ascii=False, indent=2)
print(json.dumps({
    'phase': 1, 'step': 4, 'operation': 'prepare_mcr004_historical_regression',
    'git_commit': commit, 'algorithm_version': summary['algorithm_version'],
    'status': summary['status'], 'expected_unique_studies': summary['expected_unique_studies'],
    'expected_knees': summary['expected_knees'],
    'candidate_knees': summary['candidate_knees'],
    'abstained_knees': summary['abstained_knees'],
    'visual_review_status': summary['visual_review_status'],
    'mass_processing_executed': False, 'partitions_created': False,
    'training_executed': False, 'reserved_test_opened': False,
}, ensure_ascii=False, indent=2))
print('Revisión técnica pendiente en:', output_dir/'revision_tecnica_ciega.csv')

## Mostrar las candidatas para revisión técnica

Comparar cada caja cian y su recorte con la mitad original. Registrar las decisiones en el CSV privado siguiendo MCR-2026-004, sin consultar desenlaces ni reclasificar abstenciones como casos correctos. Los criterios y umbrales ya están fijados.

In [ ]:
import csv
from IPython.display import display
from PIL import Image
review_path = output_dir/'revision_tecnica_ciega.csv'
with review_path.open(newline='', encoding='utf-8-sig') as stream:
    rows = list(csv.DictReader(stream, delimiter=';'))
if len(rows) != 20:
    raise RuntimeError('El denominador de revisión no es 20.')
for row in rows:
    print(row['knee_alias'], row['candidate_status'])
    if row['preview_file']:
        display(Image.open(output_dir/row['preview_file']))
print('No se cierra el paso 4 con esta celda. Queda pendiente la revisión de las 20 filas.')